### 06.03. Setup functions and indexes

In [1]:
#Install prerequisite packages
!pip install python-dotenv==1.0.0

!pip install llama-index==0.10.59
!pip install llama-index-core==0.10.59
!pip install llama-index-llms-gemini==0.1.11
!pip install llama-index-embeddings-gemini==0.1.8



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
#Setup Gemini connection
from dotenv import load_dotenv
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core import Settings
from google.api_core.exceptions import ResourceExhausted
import os
import nest_asyncio
import re
import time

nest_asyncio.apply()
load_dotenv()

# Gemini API key. Set this in your environment or in a local .env file:
# GEMINI_API_KEY=your_api_key_here
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not gemini_api_key:
    raise ValueError("Set GEMINI_API_KEY in your environment or local .env file before running this notebook.")

# The LlamaIndex Gemini integration reads GOOGLE_API_KEY internally.
os.environ["GOOGLE_API_KEY"] = gemini_api_key

gemini_model = os.getenv("GEMINI_MODEL", "models/gemini-2.5-flash")
if not (gemini_model.startswith("models/") or gemini_model.startswith("tunedModels/")):
    gemini_model = f"models/{gemini_model}"

gemini_embed_model = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-2")
if not gemini_embed_model.startswith("models/"):
    gemini_embed_model = f"models/{gemini_embed_model}"

is_embedding_2_model = "gemini-embedding-2" in gemini_embed_model

_gemini_last_request_at = 0.0


def _wait_for_gemini_rate_limit():
    global _gemini_last_request_at
    min_interval = float(os.getenv("GEMINI_MIN_REQUEST_INTERVAL_SECONDS", "13"))
    elapsed = time.monotonic() - _gemini_last_request_at
    wait_time = min_interval - elapsed
    if wait_time > 0:
        time.sleep(wait_time)
    _gemini_last_request_at = time.monotonic()


def _quota_retry_delay_seconds(error):
    message = str(error)
    retry_match = re.search(r"Please retry in ([0-9.]+)s", message)
    if retry_match:
        return float(retry_match.group(1)) + 2
    retry_delay_match = re.search(r"retry_delay\\s*\\{\\s*seconds:\\s*(\\d+)", message)
    if retry_delay_match:
        return float(retry_delay_match.group(1)) + 2
    return 60.0


class RateLimitedGemini(Gemini):
    def _call_with_quota_retry(self, call):
        max_retries = int(os.getenv("GEMINI_QUOTA_RETRIES", "2"))
        for attempt in range(max_retries + 1):
            _wait_for_gemini_rate_limit()
            try:
                return call()
            except ResourceExhausted as error:
                if attempt >= max_retries:
                    raise
                wait_time = _quota_retry_delay_seconds(error)
                print(f"Gemini quota reached. Waiting {wait_time:.1f}s before retrying...")
                time.sleep(wait_time)

    def chat(self, messages, **kwargs):
        return self._call_with_quota_retry(lambda: super(RateLimitedGemini, self).chat(messages, **kwargs))

    def complete(self, prompt, formatted=False, **kwargs):
        return self._call_with_quota_retry(lambda: super(RateLimitedGemini, self).complete(prompt, formatted=formatted, **kwargs))


#Setup the LLM
Settings.llm = RateLimitedGemini(
    model=gemini_model,
    api_key=gemini_api_key,
    temperature=0.1,
)

#Setup the embedding model for RAG
Settings.embed_model = GeminiEmbedding(
    model_name=gemini_embed_model,
    api_key=gemini_api_key,
    task_type=None if is_embedding_2_model else "retrieval_document",
)


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from typing import List
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import  VectorStoreIndex
from llama_index.core.tools import QueryEngineTool

#-------------------------------------------------------------
# Tool 1 : Function that returns the list of items in an order
#-------------------------------------------------------------
def get_order_items(order_id: int) -> List[str] :
    """Given an order Id, this function returns the 
    list of items purchased for that order"""
    
    order_items = {
            1001: ["Laptop","Mouse"],
            1002: ["Keyboard","HDMI Cable"],
            1003: ["Laptop","Keyboard"]
        }
    if order_id in order_items.keys():
        return order_items[order_id]
    else:
        return []

#-------------------------------------------------------------
# Tool 2 : Function that returns the delivery date for an order
#-------------------------------------------------------------
def get_delivery_date(order_id: int) -> str:
    """Given an order Id, this function returns the 
    delivery date for that order"""

    delivery_dates = {
            1001: "10-Jun",
            1002: "12-Jun",
            1003: "08-Jun"       
    }
    if order_id in delivery_dates.keys():
        return delivery_dates[order_id]
    else:
        return []

#----------------------------------------------------------------
# Tool 3 : Function that returns maximum return days for an item
#----------------------------------------------------------------
def get_item_return_days(item: str) -> int :
    """Given an Item, this function returns the return support
    for that order. The return support is in number of days"""
    
    item_returns = {
            "Laptop"     : 30,
            "Mouse"      : 15,
            "Keyboard"   : 15,
            "HDMI Cable" : 5
    }
    if item in item_returns.keys():
        return item_returns[item]
    else:
        #Default
        return 45

#-------------------------------------------------------------
# Tool 4 : Vector DB that contains customer support contacts
#-------------------------------------------------------------
#Setup vector index for return policies
support_docs=SimpleDirectoryReader(input_files=["Customer Service.pdf"]).load_data()

splitter=SentenceSplitter(chunk_size=1024)
support_nodes=splitter.get_nodes_from_documents(support_docs)
support_index=VectorStoreIndex(support_nodes)
support_query_engine = support_index.as_query_engine()


### 06.04. Setup the Customer Service AI Agent

In [4]:
from llama_index.core.tools import FunctionTool

#Create tools for the 3 functions and 1 index
order_item_tool = FunctionTool.from_defaults(fn=get_order_items)
delivery_date_tool = FunctionTool.from_defaults(fn=get_delivery_date)
return_policy_tool = FunctionTool.from_defaults(fn=get_item_return_days)

support_tool = QueryEngineTool.from_defaults(
    query_engine=support_query_engine,
    description=(
        "Customer support policies and contact information"
    ),
)

In [5]:
from llama_index.core.agent import ReActAgent

#Setup the Agent in LlamaIndex with all the Tools.
#Gemini is used through a ReAct agent because this LlamaIndex Gemini wrapper
#does not expose the native function-calling interface required by FunctionCallingAgentWorker.
agent = ReActAgent.from_tools(
    [order_item_tool,
     delivery_date_tool,
     return_policy_tool,
     support_tool
    ],
    llm=Settings.llm,
    verbose=True,
    max_iterations=10,
)


### 06.05. Using the customer service Agent

In [6]:
#Get return policy for an order
response = agent.query(
    "What is the return policy for order number 1001"
)

print("\n Final output : \n", response)

> Running step c18753aa-236e-448a-a6ac-904aef12ad9f. Step input: What is the return policy for order number 1001
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_order_items
Action Input: {'order_id': 1001}
Observation: ['Laptop', 'Mouse']
> Running step e5fb5bf8-6b19-43bf-adce-1909f72acdf0. Step input: None
Thought: The current language of the user is: English. I have the items for order 1001. Now I need to find the return policy for each item. I will start with 'Laptop'.
Action: get_item_return_days
Action Input: {'item': 'Laptop'}
Observation: 30
> Running step 7993fee5-51e1-430c-ae7b-3ab2f6ab77dc. Step input: None
Thought: The current language of the user is: English. I have the return policy for 'Laptop'. Now I need to find the return policy for 'Mouse'.
Action: get_item_return_days
Action Input: {'item': 'Mouse'}
Observation: 15
> Running step 5d3f40d5-7bbc-4b21-9527-80cb63f404b2. Step input: None
Thought: The

In [7]:
# Three part question
response = agent.query(
    "When is the delivery date and items shipped for order 1003 and how can I contact customer support?"
)

print("\n Final output : \n", response)

> Running step f52f86a5-dddb-4995-a8d3-814fe28af7c3. Step input: When is the delivery date and items shipped for order 1003 and how can I contact customer support?
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_delivery_date
Action Input: {'order_id': 1003}
Observation: 08-Jun
> Running step aa43ef47-87d4-4a42-90bf-fbcd04a07545. Step input: None
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_order_items
Action Input: {'order_id': 1003}
Observation: ['Laptop', 'Keyboard']
> Running step 4d77bd3d-ac28-46c4-a750-72adf700d691. Step input: None
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: query_engine_tool
Action Input: {'input': 'how can I contact customer support?'}
Observation: You can contact customer support by calling 1-987-654-3210 or by sending an email to support@compa

In [8]:
#Question about an invalid order number
response = agent.query(
    "What is the return policy for order number 1004"
)

print("\n Final output : \n", response)

> Running step 537c843d-94dd-4a61-bee1-024b2f5a725b. Step input: What is the return policy for order number 1004
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_order_items
Action Input: {'order_id': 1004}
Observation: []
> Running step 0bf48927-8030-44b1-8f12-cbef46d55928. Step input: None
Thought: The current language of the user is: English. The previous tool call returned an empty list of items for order 1004. This means there are no items associated with that order, or the order ID is invalid. I cannot determine the return policy without knowing the items. I should inform the user that the order ID might be invalid or has no items.
Answer: I could not find any items for order number 1004. Please check if the order number is correct.

 Final output : 
 I could not find any items for order number 1004. Please check if the order number is correct.
